Importación de paquetes

In [ ]:
import cv2  
import math 
from pathlib import Path
import json
from collections import defaultdict
import numpy as np

from ultralytics import YOLO


Entrenamiento de matrículas 

In [3]:
DATASET = Path("dataset").resolve()

# Convierte los labels .json a .txt de YOLO (clase 0 = matrícula)
for split in ["train", "val", "test"]:
    for js in (DATASET / split / "labels").glob("*.json"):
        boxes = json.loads(js.read_text(encoding="utf-8"))
        lines = []
        for b in boxes:
            x, y, w, h = (min(max(v, 0.0), 1.0) for v in b["rectangle"])
            lines.append(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
        js.with_suffix(".txt").write_text("\n".join(lines), encoding="utf-8")

# Configuración del dataset
(DATASET / "data.yaml").write_text(
    f"path: {DATASET.as_posix()}\n"
    "train: train/images\n"
    "val: val/images\n"
    "test: test/images\n"
    "names:\n"
    "  0: plate\n",
    encoding="utf-8",
)

# Entrenamiento
model = YOLO("yolo11n.pt")
results = model.train(
    data=str(DATASET / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=960,
    batch=4,
    project=str(Path.cwd() / "results"),
    name="plate-detection-augmentation",
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    degrees=5.0, translate=0.1, scale=0.5, shear=2.0, perspective=0.0005,
    fliplr=0.0,
    mosaic=1.0, mixup=0.1,
    erasing=0.3,
    close_mosaic=10
)

# Evaluación sobre el conjunto de test
metrics = model.val(data=str(DATASET / "data.yaml"), split="test")

New https://pypi.org/project/ultralytics/8.4.203 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.200  Python-3.9.25 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Dylan\Desktop\Año 5\VC\VC_P4\dataset\data.yaml, degrees=5.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.3, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.6, hsv_v=0.5, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

Prueba del modelo en video

In [4]:
VIDEO_IN = "TGC23_PdH_C0056cut.mp4"
VIDEO_OUT = Path("results") / "plate-detection-augmentation" / "video_detection.mp4"

# Pesos del modelo entrenado
plate_model = YOLO(str(Path("results") / "plate-detection-augmentation" / "weights" / "best.pt"))

cap = cv2.VideoCapture(VIDEO_IN)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(VIDEO_OUT), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Detección de matrículas (mismo imgsz que en el entrenamiento)
    res = plate_model.predict(frame, imgsz=960, conf=0.25, verbose=False)[0]

    for box in res.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"plate {conf:.2f}", (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    writer.write(frame)
    cv2.imshow("Matriculas", frame)
    # Detenemos pulsando ESC
    if cv2.waitKey(1) == 27:
        break

cap.release()
writer.release()
cv2.destroyAllWindows()
print("Vídeo guardado en:", VIDEO_OUT)

Vídeo guardado en: results\plate-detection-augmentation\video_detection.mp4


Entrenamientos para lector de matrículas

Paddle